# Deteksi Kerusakan Jalan · YOLO di VS Code (lokal)

Versi lokal dari notebook Colab. Dataset dan hasil training disimpan di folder project ini, jadi tidak perlu Google Drive dan tidak perlu ekstrak ulang tiap sesi.

**Persiapan (sekali saja):**
1. Install **Python 3.12** dari python.org. Saat install, centang **Add python.exe to PATH**.
2. Install **VS Code**, lalu pasang extension **Python** dan **Jupyter** (keduanya dari Microsoft).
3. Kalau punya GPU NVIDIA, update driver-nya ke versi terbaru (lewat NVIDIA App atau nvidia.com).
4. Buat folder project **di luar OneDrive**, misalnya `C:\deteksi_jalan` atau `D:\deteksi_jalan`, lalu taruh notebook ini di dalamnya. Di VS Code: **File → Open Folder** → pilih folder itu.
5. Buka terminal (**Terminal → New Terminal**), jalankan `python -m venv .venv` (kalau `python` tidak dikenali, pakai `py -m venv .venv`).
6. Buka notebook ini, klik **Select Kernel** di kanan atas → **Python Environments** → pilih **.venv**. Kalau VS Code minta pasang `ipykernel`, klik **Install**.
7. Jalankan **Sel 0** sekali, lalu klik **Restart** di toolbar notebook.

**Cara pakai:**
- **Sel 1 (Setup):** jalankan tiap kali buka VS Code atau restart kernel.
- **Bagian A (Persiapan data):** cukup sekali. Jalankan berurutan dari A1 sampai A9.
- **Bagian C (Training):** pilih model dan skema di C1, lalu jalankan C1 sampai C4. Kalau training terhenti, jalankan Sel 1 → C1 → C3 lagi, otomatis lanjut dari checkpoint terakhir.
- **Bagian D (Evaluasi):** setelah training selesai. Hasilnya ditambahkan ke `hasil_evaluasi.csv`.

**Isi folder project nanti:**
```
deteksi_jalan/
├── deteksi_kerusakan_jalan_vscode.ipynb
├── data/
│   ├── rddc2024id.zip         dataset asli
│   ├── raw/                   hasil ekstrak zip
│   └── rddc/                  dataset siap training + data.yaml
├── runs/<nama eksperimen>/    best.pt, last.pt, log, grafik
├── models/                    salinan best/last tiap eksperimen
├── eval/                      hasil evaluasi data uji
└── hasil_evaluasi.csv
```
Butuh ruang disk kira-kira 3 kali ukuran zip dataset, ditambah sekitar 3 GB untuk PyTorch versi GPU.

In [4]:
# Sel 0 · Instalasi library (cukup sekali, jalankan sebelum sel lain lalu restart kernel)
import sys, shutil, subprocess

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)

ada_gpu_nvidia = shutil.which("nvidia-smi") is not None
cek = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                     capture_output=True, text=True)   # dicek di proses terpisah supaya file torch tidak terkunci
torch_terpasang = cek.returncode == 0
torch_cuda = cek.stdout.strip() == "True"

if ada_gpu_nvidia and not torch_cuda:
    print("GPU NVIDIA terdeteksi, memasang PyTorch versi CUDA (unduhan sekitar 3 GB)...")
    if torch_terpasang:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision"], check=True)
    pip("torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu128")
elif not torch_terpasang:
    print("GPU NVIDIA tidak terdeteksi, memasang PyTorch versi CPU.")
    pip("torch", "torchvision")
else:
    print("PyTorch sudah terpasang", "(GPU siap)" if torch_cuda else "(CPU)")

pip("ultralytics==8.4.190", "albumentations>=1.4.22", "imagehash", "gdown",
    "pandas", "scikit-learn", "matplotlib", "tqdm", "ipywidgets")
print("\nSelesai. Klik Restart di toolbar notebook, lalu lanjut ke Sel 1.")

GPU NVIDIA terdeteksi, memasang PyTorch versi CUDA (unduhan sekitar 3 GB)...
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached torch-2.11.0%2Bcu128-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0%2Bcu128-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_bindings-12.9.7-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 43.2 MB/s  0:00:14:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 49.9 MB/s  0:00:05:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 49.4 MB/s  0:00:06:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 50.6 MB/s  0:00:02ta 0:00:01
  Usin

Resuming download torch-2.11.0%2Bcu128-cp310-cp310-manylinux_2_28_x86_64.whl (607.1 MB/820.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━ 607.1/820.2 MB ? eta -:--:--0m
Resuming download torch-2.11.0%2Bcu128-cp310-cp310-manylinux_2_28_x86_64.whl (607.1 MB/820.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━ 607.1/820.2 MB ? eta -:--:--0m
Resuming download torch-2.11.0%2Bcu128-cp310-cp310-manylinux_2_28_x86_64.whl (607.1 MB/820.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━ 607.1/820.2 MB ? eta -:--:--0m
Resuming download torch-2.11.0%2Bcu128-cp310-cp310-manylinux_2_28_x86_64.whl (607.1 MB/820.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━ 607.1/820.2 MB ? eta -:--:--0m
Resuming download torch-2.11.0%2Bcu128-cp310-cp310-manylinux_2_28_x86_64.whl (607.1 MB/820.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━ 607.1/820.2 MB ? eta -:--:--0m


error: incomplete-download

× Download failed after 6 attempts because not enough bytes were received (607.1 MB/820.2 MB)
╰─> URL: https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp310-cp310-manylinux_2_28_x86_64.whl#sha256=72d53f3176a69cc20710c4ecb95f7dc4c6ba10c4e4eda45b8396ee79ee40f75a

note: This is an issue with network connectivity, not pip.
hint: Use --resume-retries to configure resume attempt limit.


CalledProcessError: Command '['/home/galang/skripsi-training/.venv-tf/bin/python', '-m', 'pip', 'install', 'torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128']' returned non-zero exit status 1.

In [ ]:
# Sel 1 · Setup (jalankan tiap buka VS Code atau restart kernel)
import os, sys, glob, shutil, random, zipfile, collections
PROJ = os.path.abspath(os.environ.get("PROJ_DIR", os.getcwd()))   # folder project = folder tempat notebook ini
DATA_DIR = os.path.join(PROJ, "data")
RAW, DS = os.path.join(DATA_DIR, "raw"), os.path.join(DATA_DIR, "rddc")
RUNS = os.path.join(PROJ, "runs")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RUNS, exist_ok=True)

import torch, ultralytics
print("Folder project:", PROJ)
print("Ultralytics", ultralytics.__version__, "| PyTorch", torch.__version__)
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name} | VRAM {g.total_memory / 1024**3:.1f} GB")
else:
    print("PERINGATAN: GPU NVIDIA tidak terdeteksi, training akan jalan di CPU dan bisa makan waktu berhari-hari.")

import numpy as np, pandas as pd
from PIL import Image
from IPython.display import display
CLASS_NAMES = ["Longitudinal Crack", "Lateral Crack", "Alligator Crack", "Pothole", "Other"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
SEED = 42
random.seed(SEED); np.random.seed(SEED)

---
## Bagian A · Persiapan data (cukup sekali)

Kalau file `data/rddc/data.yaml` sudah ada, bagian ini sudah pernah dijalankan. Langsung ke Bagian C.

In [ ]:
# A1 · Ambil foto RDDC2024-ID: unduh zip (sekali), lalu ekstrak ke data/raw
# Sumber: link Google Drive di README github.com/endurancechallengeindonesia/RDDC2024-ID
RDDC_URL = "https://drive.google.com/file/d/1AbNe-dhK2ikaK05ATsPUcJXWW_bsz-pp/view"
zip_drive = os.path.join(DATA_DIR, "rddc2024id.zip")
if not (os.path.exists(zip_drive) and zipfile.is_zipfile(zip_drive)):
    import gdown
    try:
        gdown.download(id="1AbNe-dhK2ikaK05ATsPUcJXWW_bsz-pp", output=zip_drive, quiet=False)
    except Exception as e:
        print("gdown gagal:", e)
if not (os.path.exists(zip_drive) and zipfile.is_zipfile(zip_drive)):
    raise RuntimeError(f"Unduhan gagal (biasanya kuota unduh Google Drive lagi penuh).\n"
                       f"Unduh manual dari {RDDC_URL}\nlalu simpan sebagai {zip_drive} dan jalankan sel ini lagi.")

def daftar_foto(root):
    return [p for p in glob.glob(os.path.join(root, "**", "*"), recursive=True)
            if os.path.splitext(p)[1].lower() in IMG_EXT]

if not daftar_foto(RAW):
    shutil.rmtree(RAW, ignore_errors=True); os.makedirs(RAW)
    with zipfile.ZipFile(zip_drive) as z:
        z.extractall(RAW)
    for inner in glob.glob(os.path.join(RAW, "**", "*.zip"), recursive=True):   # kalau ada zip di dalam zip
        with zipfile.ZipFile(inner) as z:
            z.extractall(os.path.splitext(inner)[0])
        os.remove(inner)

for d, _, files in sorted(os.walk(RAW)):
    depth = d[len(RAW):].count(os.sep)
    if depth <= 2:
        print("  " * depth + (os.path.basename(d) or "raw") + "/", f"({len(files)} file)")
print("Total foto:", len(daftar_foto(RAW)))

In [ ]:
# A2 · Rapikan ke struktur Ultralytics (images/ dan labels/)
# Foto dan label diambil dari data/raw (hasil ekstrak A1), apa pun nama foldernya.
semua = glob.glob(os.path.join(RAW, "**", "*"), recursive=True)
images = sorted(p for p in semua if os.path.splitext(p)[1].lower() in IMG_EXT)
if not images:
    raise FileNotFoundError(f"Belum ada foto di {RAW}. Jalankan sel A1 dulu. "
                            "A1 akan mengunduh dan mengekstrak dataset ke folder data/raw.")
label_map = {}
for p in sorted((p for p in semua if p.lower().endswith(".txt")),
                key=lambda p: "label" not in p.lower()):   # file di folder Labels didahulukan
    label_map.setdefault(os.path.splitext(os.path.basename(p))[0], p)

img_dir, lbl_dir = os.path.join(DS, "images"), os.path.join(DS, "labels")
os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True)

rows, missing, seen, dup = [], 0, set(), 0
for p in images:
    stem = os.path.splitext(os.path.basename(p))[0]
    if stem in seen:
        dup += 1
        continue
    seen.add(stem)
    dst_img = os.path.join(img_dir, os.path.basename(p))
    if not os.path.exists(dst_img):
        shutil.copy2(p, dst_img)
    dst_lbl = os.path.join(lbl_dir, stem + ".txt")
    if stem in label_map:
        shutil.copy2(label_map[stem], dst_lbl)
    else:
        missing += 1
        open(dst_lbl, "w").close()   # tanpa label = foto tanpa kerusakan
    rows.append({"stem": stem, "image": dst_img, "label": dst_lbl})

df = pd.DataFrame(rows)
print(f"Jumlah foto: {len(df)} | foto tanpa file label: {missing}" + (f" | nama kembar dilewati: {dup}" if dup else ""))

In [ ]:
# A3 · Audit label (baris rusak dibuang dari salinan label)
issues, cls_counts, n_boxes = [], collections.Counter(), []
for r in df.itertuples():
    good_lines = []
    with open(r.label) as f:
        lines = [l.strip() for l in f if l.strip()]
    for i, line in enumerate(lines, 1):
        parts = line.split(); problem = None
        if len(parts) != 5:
            problem = "jumlah kolom bukan 5"
        else:
            try:
                c = int(float(parts[0])); x, y, w, h = map(float, parts[1:])
                if c < 0 or c >= len(CLASS_NAMES): problem = f"kelas {c} di luar 0-{len(CLASS_NAMES)-1}"
                elif not all(0 <= v <= 1 for v in (x, y, w, h)): problem = "koordinat di luar 0-1"
                elif w <= 0 or h <= 0: problem = "lebar/tinggi kotak nol"
            except ValueError:
                problem = "bukan angka"
        if problem:
            issues.append({"stem": r.stem, "baris": i, "isi": line, "masalah": problem})
        else:
            good_lines.append(line); cls_counts[c] += 1
    n_boxes.append(len(good_lines))
    if len(good_lines) != len(lines):
        with open(r.label, "w") as f:
            f.write("\n".join(good_lines) + ("\n" if good_lines else ""))

df["n_boxes"] = n_boxes
issues_df = pd.DataFrame(issues)
issues_df.to_csv(os.path.join(PROJ, "audit_label_bermasalah.csv"), index=False)
print(f"Baris label bermasalah (dibuang): {len(issues_df)}")
if len(issues_df): display(issues_df["masalah"].value_counts())
print(f"Foto tanpa kotak (background): {(df.n_boxes == 0).sum()}")
display(pd.DataFrame({"kelas": CLASS_NAMES, "jumlah kotak": [cls_counts[i] for i in range(len(CLASS_NAMES))]}))

In [ ]:
# A4 · pHash 64-bit tiap foto + jarak ke tetangga terdekat
import imagehash
from tqdm.auto import tqdm

def phash_int(path):
    with Image.open(path) as im:
        return int(str(imagehash.phash(im.convert("RGB"))), 16)

df["phash"] = [phash_int(p) for p in tqdm(df.image, desc="pHash")]
H = df.phash.to_numpy(dtype=np.uint64)
POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def hamming_rows(a, B):
    x = np.bitwise_xor(a[:, None], B[None, :])
    return POP8[x.view(np.uint8)].reshape(len(a), len(B), 8).sum(axis=2)

nn = np.empty(len(H), dtype=np.int64)
for s in range(0, len(H), 512):
    d = hamming_rows(H[s:s+512], H).astype(np.int64)
    d[np.arange(d.shape[0]), np.arange(s, s + d.shape[0])] = 99
    nn[s:s+512] = d.min(axis=1)
df["nn_dist"] = nn
print("Jumlah foto per jarak tetangga terdekat (0-12):")
print(df.nn_dist.value_counts().sort_index().loc[:12])

In [ ]:
# A5 · Lihat contoh pasangan per rentang jarak, buat menentukan TAU di sel A6
import matplotlib.pyplot as plt
def sample_pair(lo, hi):
    cand = df.index[(df.nn_dist >= lo) & (df.nn_dist <= hi)].tolist()
    if not cand: return None
    i = random.choice(cand)
    d = hamming_rows(H[i:i+1], H)[0].astype(int); d[i] = 99
    return i, int(d.argmin()), int(d.min())

bands = [(0, 2), (3, 5), (6, 8), (9, 12)]
fig, axes = plt.subplots(len(bands), 2, figsize=(8, 3 * len(bands)))
for row, (lo, hi) in enumerate(bands):
    pair = sample_pair(lo, hi)
    for col in range(2):
        ax = axes[row, col]; ax.axis("off")
        if pair:
            ax.imshow(Image.open(df.image[pair[col]])); ax.set_title(f"jarak {pair[2]} (rentang {lo}-{hi})", fontsize=9)
plt.tight_layout(); plt.show()
print("Kalau pasangan di suatu rentang masih 'adegan yang sama', TAU minimal sampai rentang itu.")

In [ ]:
# A6 · Kelompokkan foto hampir kembar (jarak <= TAU)
TAU = 6   # ubah sesuai hasil sel A5, lalu jalankan ulang sel ini sampai A9

parent = list(range(len(H)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
for s in range(0, len(H), 512):
    d = hamming_rows(H[s:s+512], H)
    rr, cc = np.nonzero(d <= TAU)
    for r_, c_ in zip(rr + s, cc):
        if c_ > r_:
            a, b = find(int(r_)), find(int(c_))
            if a != b: parent[b] = a
df["group"] = [find(i) for i in range(len(H))]
sizes = df.group.value_counts()
print(f"TAU = {TAU}: {len(df)} foto -> {len(sizes)} kelompok | kelompok >1 foto: {(sizes > 1).sum()} | terbesar: {sizes.max()}")

In [ ]:
# A7 · Bagi 70:15:15 per kelompok, proporsi kelas dijaga
from sklearn.model_selection import StratifiedGroupKFold

def rarest_class(label_path):
    with open(label_path) as f:
        cs = {int(float(l.split()[0])) for l in f if l.strip()}
    return min(cs, key=lambda c: cls_counts[c]) if cs else len(CLASS_NAMES)

df["strat"] = [rarest_class(p) for p in df.label]
fold = np.empty(len(df), dtype=int)
for k, (_, idx) in enumerate(StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
                             .split(df, df.strat, df.group)):
    fold[idx] = k
df["split"] = np.where(fold < 3, "test", np.where(fold < 6, "val", "train"))
assert df.groupby("group").split.nunique().max() == 1, "Ada kelompok yang terbelah!"

def write_split(frame, prefix, yaml_name):
    for s in ["train", "val", "test"]:
        with open(os.path.join(DS, f"{prefix}{s}.txt"), "w") as f:
            f.write("\n".join(frame.loc[frame.split == s, "image"]) + "\n")
    with open(os.path.join(DS, yaml_name), "w") as f:
        f.write(f"path: '{DS.replace(os.sep, '/')}'\ntrain: {prefix}train.txt\nval: {prefix}val.txt\ntest: {prefix}test.txt\n")
        f.write("names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES)))

def class_table(frame):
    out = {}
    for s in ["train", "val", "test"]:
        cnt = collections.Counter()
        for p in frame.loc[frame.split == s, "label"]:
            with open(p) as f:
                for l in f:
                    if l.strip(): cnt[int(float(l.split()[0]))] += 1
        out[s] = [cnt[i] for i in range(len(CLASS_NAMES))]
    t = pd.DataFrame(out, index=CLASS_NAMES)
    t.loc["Jumlah foto"] = [int((frame.split == s).sum()) for s in ["train", "val", "test"]]
    return t

write_split(df, "", "data.yaml")
display(class_table(df))

In [ ]:
# A8 · (Opsional) Pembagian acak tanpa pengelompokan, buat membandingkan efek kebocoran data
rnd = df.copy()
perm = np.random.RandomState(SEED).permutation(len(rnd))
n15 = int(0.15 * len(rnd))
rnd["split"] = "train"
rnd.iloc[perm[:n15], rnd.columns.get_loc("split")] = "test"
rnd.iloc[perm[n15:2 * n15], rnd.columns.get_loc("split")] = "val"
write_split(rnd, "random_", "data_random.yaml")
print(f"Pada pembagian acak, {(rnd.groupby('group').split.nunique() > 1).sum()} kelompok foto kembar terbelah.")

In [ ]:
# A9 · Simpan ringkasan pembagian data
df.drop(columns=["phash"]).to_csv(os.path.join(PROJ, "split_kelompok.csv"), index=False)
n_img = len(os.listdir(os.path.join(DS, "images")))
print(f"Dataset siap di {DS}: {n_img} foto. Bagian A selesai, lanjut ke Bagian C.")
print("Kalau disk mepet, folder data/raw boleh dihapus sekarang (zip aslinya masih ada).")

---
## Bagian C · Training

Daftar eksperimen. Ganti `MODEL` dan `SKEMA` di sel C1, lalu jalankan C1 sampai C4. Satu eksperimen bisa makan beberapa jam (tergantung GPU), jadi colok charger dan atur supaya laptop tidak *sleep*.

Pengaturan: maksimal 15 epoch, *early stopping* (berhenti kalau 5 epoch berturut-turut tidak membaik). Tiap eksperimen menghasilkan:
- `best.pt`: model di epoch dengan mAP50-95 validasi tertinggi
- `last.pt`: model di epoch terakhir yang dijalankan
- `log_per_epoch.txt` dan `results.csv`: detail tiap epoch
- `grafik_per_epoch.png`: grafik loss dan metrik

Semuanya ada di `runs/<nama eksperimen>/`, dan salinan kedua model ada di `models/`.

| No | MODEL | SKEMA | Selesai |
|---|---|---|---|
| 1 | yolov8s.pt | A | ☐ |
| 2 | yolov8s.pt | B | ☐ |
| 3 | yolo11s.pt | A | ☐ |
| 4 | yolo11s.pt | B | ☐ |
| 5 | yolo26s.pt | A | ☐ |
| 6 | yolo26s.pt | B | ☐ |
| Opsional | yolo11s.pt | A, `SPLIT = "acak"` | ☐ |

Skema A = augmentasi bawaan YOLO. Skema B = bawaan + kecerahan, kontras, silau, bayangan.

In [ ]:
# C1 · Pilih eksperimen
MODEL = "yolo11s.pt"     # "yolov8s.pt", "yolo11s.pt", atau "yolo26s.pt"
SKEMA = "A"              # "A" = augmentasi bawaan, "B" = bawaan + augmentasi cuaca
SPLIT = "kelompok"       # "kelompok" (utama) atau "acak" (opsional, pembanding kebocoran)

import albumentations as A
NAME = f"{os.path.splitext(MODEL)[0]}_{SKEMA}" + ("_acak" if SPLIT == "acak" else "")
DATA = os.path.join(DS, "data.yaml" if SPLIT == "kelompok" else "data_random.yaml")

EPOCHS = 15        # jumlah epoch maksimum
PATIENCE = 5       # early stopping: berhenti kalau 5 epoch berturut-turut mAP50-95 validasi tidak naik
CLOSE_MOSAIC = 3   # mosaic dimatikan di 3 epoch terakhir (bawaan 10, terlalu banyak untuk 15 epoch)

BATCH = 16         # kalau muncul "CUDA out of memory", turunkan ke 8 (lalu pakai 8 juga untuk semua eksperimen)
WORKERS = 4        # proses pembaca data. Kalau error DataLoader atau "paging file is too small", ganti ke 2 atau 0

COMMON = dict(data=DATA, epochs=EPOCHS, patience=PATIENCE, close_mosaic=CLOSE_MOSAIC,
              imgsz=640, batch=BATCH, workers=WORKERS, seed=SEED, optimizer="auto",
              project=RUNS, name=NAME, exist_ok=True)

AUG_B = [
    A.Blur(p=0.01), A.MedianBlur(p=0.01), A.ToGray(p=0.01), A.CLAHE(p=0.01),   # bawaan Ultralytics, tetap dipertahankan
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),  # kecerahan dan kontras
        A.RandomSunFlare(flare_roi=(0, 0, 1, 0.5)),                            # silau dari arah langit
        A.RandomShadow(shadow_roi=(0, 0.5, 1, 1)),                              # bayangan di area jalan
    ], p=0.5),
]
print("Eksperimen:", NAME, "| data:", os.path.basename(DATA))

In [ ]:
# C2 · (Opsional) Lihat contoh hasil augmentasi Skema B sebelum training
import matplotlib.pyplot as plt
samples = random.sample(open(os.path.join(DS, "train.txt")).read().splitlines(), 2)
previews = [("Asli", None),
            ("Kecerahan/kontras", A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1)),
            ("Silau", A.RandomSunFlare(flare_roi=(0, 0, 1, 0.5), p=1)),
            ("Bayangan", A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), p=1))]
fig, axes = plt.subplots(len(samples), len(previews), figsize=(4 * len(previews), 3 * len(samples)))
for r, path in enumerate(samples):
    img = np.array(Image.open(path).convert("RGB"))
    for c, (title, t) in enumerate(previews):
        ax = axes[r, c]; ax.axis("off"); ax.set_title(title, fontsize=10)
        ax.imshow(img if t is None else t(image=img)["image"])
plt.tight_layout(); plt.show()

In [ ]:
# C3 · Training: early stopping, checkpoint best/last, log per epoch
import time, datetime, torch
from ultralytics import YOLO

RUN_DIR = os.path.join(RUNS, NAME)
MODELS = os.path.join(PROJ, "models")        # salinan model dengan nama jelas
os.makedirs(MODELS, exist_ok=True)

def log_write(tr, text):
    print(text)
    with open(os.path.join(tr.save_dir, "log_per_epoch.txt"), "a", encoding="utf-8") as f:
        f.write(text + "\n")

def log_header(tr):
    log_write(tr, f"=== {NAME} | mulai {datetime.datetime.now():%Y-%m-%d %H:%M} | epoch {tr.start_epoch + 1}-{tr.epochs} "
                  f"| patience {tr.args.patience} | optimizer {type(tr.optimizer).__name__} ===")

def log_epoch(tr):
    ep, m = tr.epoch + 1, tr.metrics or {}
    if getattr(tr, "_log_selesai", False):   # dipanggil lagi oleh validasi akhir best.pt setelah training
        log_write(tr, f"Validasi akhir best.pt (epoch {tr.stopper.best_epoch}): "
                      f"P {m.get('metrics/precision(B)', 0):.4f} R {m.get('metrics/recall(B)', 0):.4f} "
                      f"mAP50 {m.get('metrics/mAP50(B)', 0):.4f} mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.4f}")
        return
    losses = tr.label_loss_items(tr.tloss, prefix="train")
    is_best = tr.fitness is not None and tr.best_fitness is not None and tr.fitness >= tr.best_fitness
    line = (f"Epoch {ep:>2}/{tr.epochs} | {tr.epoch_time:5.0f} dtk | "
            + " ".join(f"{k.split('/')[-1]} {float(v):.4f}" for k, v in losses.items())
            + f" | P {m.get('metrics/precision(B)', 0):.4f} R {m.get('metrics/recall(B)', 0):.4f}"
            + f" mAP50 {m.get('metrics/mAP50(B)', 0):.4f} mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.4f}"
            + f" | lr {tr.lr.get('lr/pg0', 0):.6f}"
            + (" | TERBAIK -> best.pt" if is_best else f" | terbaik masih epoch {tr.stopper.best_epoch}"))
    log_write(tr, line)
    if tr.stop:
        tr._log_selesai = True
        if ep < tr.epochs:
            log_write(tr, f">>> Early stopping: {tr.args.patience} epoch tanpa peningkatan, training berhenti di epoch {ep}.")

def with_logging(model):
    model.add_callback("on_train_start", log_header)
    model.add_callback("on_fit_epoch_end", log_epoch)
    return model

def simpan_model():
    w = os.path.join(RUN_DIR, "weights")
    for src, tag in [("best.pt", "best"), ("last.pt", "last")]:
        shutil.copy2(os.path.join(w, src), os.path.join(MODELS, f"{NAME}_{tag}.pt"))
    res = pd.read_csv(os.path.join(RUN_DIR, "results.csv")); res.columns = res.columns.str.strip()
    b = res["metrics/mAP50-95(B)"].idxmax()
    print(f"\nEpoch terbaik: {int(res.loc[b, 'epoch'])} dari {len(res)} epoch "
          f"(mAP50-95 validasi {res.loc[b, 'metrics/mAP50-95(B)']:.4f})")
    print("Model terbaik :", os.path.join(MODELS, f"{NAME}_best.pt"))
    print("Model terakhir:", os.path.join(MODELS, f"{NAME}_last.pt"))

last = os.path.join(RUN_DIR, "weights", "last.pt")
status = "baru"
if os.path.exists(last):
    ck = torch.load(last, map_location="cpu", weights_only=False)
    status = "selesai" if ck.get("epoch", -1) == -1 else "lanjut"

if status == "selesai":
    print(f"{NAME} sudah selesai training. Kalau mau training ulang dari nol, hapus dulu folder {RUN_DIR}")
elif status == "lanjut":
    print(f"Melanjutkan {NAME} dari checkpoint terakhir...")
    with_logging(YOLO(last)).train(resume=True)
else:
    kwargs = dict(COMMON)
    if SKEMA == "B":
        kwargs["augmentations"] = AUG_B
    with_logging(YOLO(MODEL)).train(**kwargs)
simpan_model()

In [ ]:
# C4 · Lihat log dan grafik per epoch
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
log_path = os.path.join(RUN_DIR, "log_per_epoch.txt")
if os.path.exists(log_path):
    print(open(log_path, encoding="utf-8").read())

res = pd.read_csv(os.path.join(RUN_DIR, "results.csv")); res.columns = res.columns.str.strip()
ib = res["metrics/mAP50-95(B)"].idxmax(); best_ep = int(res.loc[ib, "epoch"])
display(res.drop(columns=["time", "lr/pg1", "lr/pg2"], errors="ignore").set_index("epoch").round(4))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
for k, color in [("box_loss", "C0"), ("cls_loss", "C1"), ("dfl_loss", "C2")]:
    ax1.plot(res.epoch, res[f"train/{k}"], color=color, marker="o", ms=3, label=f"{k} latih")
    ax1.plot(res.epoch, res[f"val/{k}"], color=color, marker="o", ms=3, ls="--", label=f"{k} validasi")
ax1.set_title("Loss per epoch (garis putus-putus = validasi)")
for col, lab in [("metrics/precision(B)", "Precision"), ("metrics/recall(B)", "Recall"),
                 ("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50-95")]:
    ax2.plot(res.epoch, res[col], marker="o", ms=3, label=lab)
ax2.plot(best_ep, res.loc[ib, "metrics/mAP50-95(B)"], "k*", ms=11, label="best.pt")
ax2.set_title("Metrik validasi per epoch")
for ax in (ax1, ax2):
    ax.axvline(best_ep, color="gray", ls=":", lw=1)
    ax.set_xlabel("Epoch"); ax.grid(alpha=.3); ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax2.annotate(f"terbaik: epoch {best_ep}", xy=(best_ep, 0), xycoords=("data", "axes fraction"),
             xytext=(4, 4), textcoords="offset points", fontsize=8, color="gray")
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "grafik_per_epoch.png"), dpi=150)
plt.show()

---
## Bagian D · Evaluasi di data uji

Evaluasi memakai `best.pt`. Model ini dipilih berdasarkan data validasi, jadi data uji tetap belum pernah dipakai sama sekali selama training dan hasilnya adil.

In [ ]:
# D1 · Evaluasi model terbaik di data uji, simpan ke hasil_evaluasi.csv
from ultralytics import YOLO
best = os.path.join(RUNS, NAME, "weights", "best.pt")
m = YOLO(best).val(data=DATA, split="test", imgsz=640, batch=BATCH, workers=WORKERS,
                   project=os.path.join(PROJ, "eval"), name=NAME, exist_ok=True)

P, R = m.results_dict["metrics/precision(B)"], m.results_dict["metrics/recall(B)"]
row = {"eksperimen": NAME, "model": MODEL, "skema": SKEMA, "split": SPLIT,
       "precision": P, "recall": R, "F1": 2 * P * R / (P + R) if P + R else 0.0,
       "mAP50": m.results_dict["metrics/mAP50(B)"], "mAP50-95": m.results_dict["metrics/mAP50-95(B)"],
       "inferensi_ms_per_foto": m.speed["inference"]}
ap50 = dict(zip([int(i) for i in m.box.ap_class_index], [float(v) for v in m.box.ap50]))
for i, n in enumerate(CLASS_NAMES):
    row[f"AP50 {n}"] = ap50.get(i, float("nan"))

csv_path = os.path.join(PROJ, "hasil_evaluasi.csv")
hasil = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()
if len(hasil):
    hasil = hasil[hasil.eksperimen != NAME]
hasil = pd.concat([hasil, pd.DataFrame([row])], ignore_index=True)
hasil.to_csv(csv_path, index=False)
display(pd.DataFrame([row]).T.rename(columns={0: NAME}))

In [ ]:
# D2 · Tabel semua eksperimen yang sudah dievaluasi
display(pd.read_csv(os.path.join(PROJ, "hasil_evaluasi.csv")).round(4))

---
### Catatan
- Training bisa dihentikan kapan saja (tombol **Interrupt** / ■ di toolbar). Checkpoint tersimpan tiap epoch, jadi jalankan Sel 1 → C1 → C3 lagi untuk melanjutkan. Log per epoch tetap nyambung.
- Supaya laptop tidak *sleep* saat training: **Settings → System → Power & battery → Screen, sleep & hibernate** → set **Never** saat dicolok charger.
- `CUDA out of memory` → ubah `BATCH` di C1 jadi 8. Pakai angka yang sama untuk semua eksperimen biar perbandingannya adil.
- Error DataLoader atau `paging file is too small` → ubah `WORKERS` di C1 jadi 2 atau 0.
- GPU lama (seri GTX 10xx) tidak terbaca walaupun PyTorch CUDA sudah terpasang → di Sel 0 ganti `cu128` jadi `cu126`, lalu jalankan ulang Sel 0.
- Sel 0 gagal dengan `Access is denied` → restart kernel, lalu jalankan Sel 0 paling pertama sebelum sel lain.
- Grafik bawaan Ultralytics (kurva loss, PR curve, confusion matrix) juga tersimpan di `runs/<nama eksperimen>/`.
- Tahap berikutnya: evaluasi di BPID per kondisi cuaca (notebook terpisah).